[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

# Operações Espaciais: Histograma, Filtragem e Convolução

Este capítulo aborda o processamento de imagens diretamente no domínio espacial. Investigamos a manipulação de histogramas para realce de contraste, as operações pontuais aritméticas e lógicas, além do uso de filtros locais via convolução para suavização, redução de ruído e detecção de bordas estruturais.

## Objetivos

Ao final deste capítulo, você será capaz de:

* **Manipular intensidade e pixels:** Executar operações aritméticas e lógicas (como subtração de fundo e máscaras bit a bit) e aplicar técnicas de realce por processamento de histograma (equalização e especificação);
* **Compreender fundamentos espaciais:** Entender os conceitos de vizinhança espacial, os mecanismos de convolução/correlação e o papel dos *kernels* (máscaras);
* **Aplicar filtragem espacial de suavização:** Utilizar filtros lineares de média e Gaussiano para redução de ruído e atenuação de detalhes;
* **Aplicar filtragem espacial de realce:** Dominar o uso de filtros de nitidez e bordas, como o Laplaciano, Sobel e a técnica de *Unsharp Masking*;
* **Utilizar filtros de ordem:** Aplicar o filtro de mediana para a remoção eficaz de ruídos específicos, como o ruído do tipo sal e pimenta;
* **Resolver problemas práticos:** Combinar essas técnicas de processamento e filtragem no pré-processamento de imagens para aplicações reais.

## Configuração do Ambiente

In [ ]:
#| quarto-raw: true

import os, importlib, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Baixar morph.py se necessário
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph
importlib.reload(morph)
from morph import mm

print("✅ Ambiente pronto")

In [ ]:
#| quarto-raw: true

# Imagem de exemplo padrão
url = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
img_color = mm.read(url)
img_gray = mm.gray(img_color)
print(f"Imagem carregada: {img_gray.shape} | Tipo: {img_gray.dtype}")

## Manipulação de Intensidade: Operações Aritméticas e Lógicas

As operações pontuais modificam o valor de um pixel de forma isolada, baseando-se estritamente na intensidade do próprio pixel e, em certos casos, em uma matriz correspondente externa de mesma dimensão. 

### Operações Aritméticas

As transformações aritméticas elementares como adição, subtração, multiplicação e divisão são aplicadas diretamente pixel a pixel. No ecossistema de processamento digital de imagens, a manipulação exige atenção crítica à saturação do tipo de dado (ex: estouro do limite de `uint8` [0, 255]).

- **Subtração de Fundo:** Utilizada para isolar objetos em movimento ou remover iluminação não uniforme estática da cena. Matematicamente expressa por $g(x,y) = f(x,y) - h(x,y)$.
- **Multiplicação/Divisão:** Frequentemente aplicada para a correção de ganho do sensor ou calibração de iluminação (*flat-field correction*).

### Operações Lógicas e Máscaras

As operações bit-a-bit (*bitwise*) como `AND`, `OR`, `XOR` e `NOT` operam sobre a representação binária dos pixels. Em PDI, são ferramentas fundamentais para a aplicação de **Máscaras de Região de Interesse (ROI)**, isolando regiões espaciais de formatos complexos para filtragem direcionada.

In [ ]:
#| label: fig-03-aritmetica-logica
#| fig-cap: "Exemplo de mascaramento bit-a-bit. À esquerda a imagem original, ao centro a máscara circular e à direita a aplicação da operação lógica AND."
#| echo: true
#| output: true

# Criação de uma máscara circular preta com círculo branco central
h, w = img_gray.shape
mask = np.zeros((h, w), dtype=np.uint8)
cv2.circle(mask, (w//2, h//2), 140, 255, -1)

# Aplicação da operação lógica bitwise_and
img_masked = cv2.bitwise_and(img_gray, img_gray, mask=mask)

mm.show([img_gray, mask, img_masked], 
        titles=["Original", "Máscara Binária (ROI)", "Resultado Bitwise AND"], 
        cols=3)

## Processamento de Histograma: Realce de Contraste

O histograma de uma imagem digital com níveis de intensidade no intervalo $[0, L-1]$ é uma função discreta $h(k_r) = n_r$, onde $n_r$ representa o número de pixels com intensidade $k_r$.

### Equalização de Histograma

A equalização busca obter uma distribuição uniforme de frequências de intensidade, maximizando o contraste global da cena. A transformação baseia-se na Função de Distribuição Acumulada (CDF) normalizada:

$$
s_k = T(r_k) = (L-1) \sum_{j=0}^{k} p_r(r_j) = \frac{L-1}{M \cdot N} \sum_{j=0}^{k} n_j
$$

### Especificação de Histograma (Casamento)

Enquanto a equalização opera de forma cega gerando um histograma plano ideal, a **Especificação de Histograma** permite moldar o contraste da imagem de entrada de modo que seu histograma final assemelhe-se ao perfil de uma imagem de referência dada. É a base matemática para a normalização de iluminação em conjuntos de dados científicos.

In [ ]:
#| label: fig-03-equalizacao
#| fig-cap: "Impacto da equalização de histograma no contraste. (Esquerda) Imagem original com baixo contraste artificial; (Direita) Imagem equalizada com distribuição uniforme de tons."
#| echo: true
#| output: true

# Geração de uma imagem com baixo contraste artificial para simulação
img_low_contrast = np.clip((img_gray * 0.4) + 60, 0, 255).astype(np.uint8)

# Aplicação de equalização global via OpenCV
img_equalized = cv2.equalizeHist(img_low_contrast)

mm.show([img_low_contrast, img_equalized], 
        titles=["Baixo Contraste", "Após Equalização"], 
        cols=2)

## Introdução às Operações Espaciais

As operações espaciais atuam diretamente sobre a matriz de pixels modificando o valor de uma coordenada $(x,y)$ com base nas características matemáticas dos elementos pertencentes à sua vizinhança espacial.

Esse mapeamento local utiliza uma submatriz de dimensões reduzidas (geralmente ímpares como $3\times3$, $5\times5$), comumente denominada por sinônimos contextualizados:

- **Máscara** (filtros lógicos e segmentação);
- **Kernel** (convolução algébrica e aprendizado profundo);
- **Filtro** (ênfase na resposta em frequência);
- **Janela de Convolução** (contexto computacional móvel).

A janela varre a imagem pixel por pixel, operando linearmente ou de maneira não linear na região sob sua cobertura, gerando a matriz de saída correspondente.

## Convolução Bidimensional e Correlação

A convolução espacial e a correlação cruzada formam a espinha dorsal matemática da filtragem linear. Computacionalmente, a correlação consiste no produto escalar direto dos coeficientes da máscara com a vizinhança da imagem. A **Convolução**, por sua vez, requer o espelhamento prévio do kernel em ambos os eixos (rotação de 180°).

Dada uma imagem $f(x,y)$ de dimensões $M \times N$ e um kernel de tamanho $(2a+1) \times (2b+1)$, a equação formal da convolução bidimensional discreta é definida por:

$$
g(x,y) = f(x,y) \star w(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t) \cdot f(x-s,y-t)
$$

A inversão dos índices $(x-s, y-t)$ garante a propriedade matemática da associatividade, essencial para a composição consecutiva de múltiplos filtros lineares no domínio espacial.

::: {.callout-warning}
### Tratamento de Bordas de Imagem {.unnumbered}
Durante o deslocamento do kernel nas fronteiras físicas da imagem, os coeficientes externos carecem de pixels correspondentes. As principais estratégias de atenuação desse artefato incluem: preenchimento com zeros (`BORDER_CONSTANT`), replicação do pixel limítrofe (`BORDER_REPLICATE`) ou reflexão simétrica da borda (`BORDER_REFLECT`).
:::

## Filtragem Espacial de Suavização (Passa-Baixa)

Os filtros de suavização atenuam os componentes de alta frequência espacial da imagem, atuando na redução de ruídos de alta variância e na eliminação de falsos contornos gerados por texturas irrelevantes antes do processamento estrutural.

### Filtro de Média (Box Filter)

Substitui a intensidade central pela média aritmética uniforme dos valores contidos na vizinhança. Todos os coeficientes da máscara possuem peso idêntico e a soma é normalizada para a unidade, prevenindo alteração indesejada no brilho global da cena. Um kernel de média $3\times3$ padrão é dado por:

$$
W_{\text{média}} = \frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

O filtro de média reduz o ruído de fundo, mas introduz um efeito colateral severo: o borrão indiscriminado das bordas estruturais de interesse.

In [ ]:
#| quarto-raw: true

# Construção explícita de um kernel de média 5x5 para efeito visual acentuado
kernel_media = np.ones((5,5), dtype=np.float32) / 25
print("Kernel de Média Normalizado 5x5:\n", kernel_media)

In [ ]:
#| label: fig-03-media
#| fig-cap: "Suavização linear através do filtro de média espacial 5x5."
#| echo: true
#| output: true

img_media = cv2.filter2D(img_gray, -1, kernel_media, borderType=cv2.BORDER_REFLECT)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_media, cmap='gray')
ax[1].set_title("Suavização por Média (5x5)")
ax[1].axis('off')
plt.show()

### Filtro Gaussiano

Diferente da média uniforme, a suavização Gaussiana emprega pesos ponderados não uniformes decrescentes a partir do pixel central em conformidade com uma distribuição normal bivariada:

$$
h(x,y) = e^{-\frac{x^2+y^2}{2\sigma^2}}
$$

onde $\sigma$ denota o desvio padrão da distribuição, controlando o raio efetivo do espalhamento físico do borrão. O filtro Gaussiano apresenta propriedades físicas e matemáticas singulares: é isotropicamente invariante por rotações e não gera artefatos de oscilações espaciais espúrias (frequências fantasmas), sendo a escolha mandatória para a modelagem de pirâmides de imagens e redução de ruído térmico em sensores.

In [ ]:
#| label: fig-03-gauss
#| fig-cap: "Filtragem isotrópica Gaussiana com atenuação preservada de altas frequências."
#| echo: true
#| output: true

# Suavização Gaussiana com janela 9x9 e desvio padrão sigma calculado automaticamente
img_gauss = cv2.GaussianBlur(img_gray, (9,9), sigmaX=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_gauss, cmap='gray')
ax[1].set_title("Suavização Gaussiana (9x9)")
ax[1].axis('off')
plt.show()

## Filtros de Ordem Não Lineares: O Filtro de Mediana

Filtros espaciais não lineares baseiam-se em ordenação topológica. O **Filtro de Mediana** percorre a vizinhança, ordena de forma crescente todos os valores numéricos sob a janela e atribui o valor mediano exato ao pixel de interesse central.

### Atenuação de Ruído Impulsivo (Sal-e-Pimenta)

O ruído impulsivo manifesta-se em sensores de transmissão digital como picos isolados espúrios de intensidades extremas (valores de brilho mínimo $0$ ou máximo $255$). Filtros lineares (média e Gauss) falham ao processar esse ruído pois integram os valores aberrantes ao cálculo aritmético, borrando e expandindo o ruído espacialmente.

Como os valores de ruído sal-e-pimenta posicionam-se exclusivamente nos extremos do vetor ordenado da vizinhança, a escolha matemática da mediana elimina o ruído por completo da cena reconstruída. Adicionalmente, este filtro possui a propriedade ímpar de preservar a nitidez de transições de bordas com degraus perfeitos.

In [ ]:
#| quarto-raw: true

def add_salt_pepper(img, prob=0.03):
    """Simulador probabilístico de ruído impulsivo em malha digital."""
    noisy = img.copy()
    rnd = np.random.rand(*img.shape)
    # Distribuição equilibrada entre Sal (255) e Pimenta (0)
    noisy[rnd < prob/2] = 0
    noisy[rnd > 1 - prob/2] = 255
    return noisy

img_noise = add_salt_pepper(img_gray, prob=0.04)

In [ ]:
#| label: fig-03-mediana
#| fig-cap: "Remoção de ruído impulsivo. Comparação entre a degradação e a restauração preservando bordas pelo filtro de mediana."
#| echo: true
#| output: true

img_mediana = cv2.medianBlur(img_noise, 5)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img_noise, cmap='gray')
ax[0].set_title("Degradada: Ruído Sal e Pimenta")
ax[0].axis('off')

ax[1].imshow(img_mediana, cmap='gray')
ax[1].set_title("Restaurada: Filtro Mediana (5x5)")
ax[1].axis('off')
plt.show()

## Filtragem Espacial de Realce e Nitidez (Passa-Alta)

A filtragem de realce enfatiza os componentes de altas frequências espaciais (detalhes finos, texturas e transições de bordas abruptas), atenuando áreas de baixa variação lenta de brilho.

### Filtro Laplaciano (Derivada de Segunda Ordem)

O Laplaciano é um operador diferencial isotrópico bidimensional linear baseado em derivadas de segunda ordem, definido analiticamente por:

$$
\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}
$$

Sua aproximação discreta sobre malha retangular gera máscaras cujos sinais do coeficiente central contrastam com os vizinhos imediatos. Exemplo de máscara Laplaciana com exclusão de diagonais:

$$
K_{\text{lap}} = 
\begin{bmatrix}
 0 & -1 &  0 \\
-1 &  4 & -1 \\
 0 & -1 &  0
\end{bmatrix}
$$

### Nitidez por Unsharp Masking e High-Boost

A técnica de *Unsharp Masking* consiste em subtrair uma versão suavizada (borrada) da imagem original para isolar as altas frequências (máscara de nitidez $g_{\text{mask}}(x,y)$), adicionando-a de volta na imagem original:

1. Máscara: $g_{\text{mask}}(x,y) = f(x,y) - \bar{f}(x,y)$
2. Saída High-Boost: $g(x,y) = f(x,y) + k \cdot g_{\text{mask}}(x,y)$

Onde $k \ge 1$ pondera o ganho do realce aplicado.

In [ ]:
#| quarto-raw: true

# Kernel passa-alta de realce direto combinando identidade e Laplaciano
kernel_sharp = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)

In [ ]:
#| label: fig-03-sharp
#| fig-cap: "Amplificação de altas frequências espaciais visando nitidez estrutural."
#| echo: true
#| output: true

img_sharp = cv2.filter2D(img_gray, -1, kernel_sharp, borderType=cv2.BORDER_REFLECT)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_sharp, cmap='gray')
ax[1].set_title("Realce de Nitidez (Sharp)")
ax[1].axis('off')
plt.show()

## Detecção de Bordas por Gradientes Espaciais

Bordas estruturais demarcam limites de descontinuidade física ou mudanças bruscas de refletância luminosa local na imagem, contendo as principais informações de forma de uma cena.

### O Operador de Sobel

O operador de Sobel computa uma aproximação digital da primeira derivada do gradiente espacial, incorporando uma suavização unidimensional ortogonal interna para mitigar ruídos espúrios de alta frequência. O cálculo avalia a magnitude do gradiente $|\nabla f|$ combinando duas componentes ortogonais parciais:

$$
G_x = 
\begin{bmatrix}
-1 & 0 & 1 \\
-2 & 0 & 2 \\
-1 & 0 & 1
\end{bmatrix}
, \quad
G_y = 
\begin{bmatrix}
-1 & -2 & -1 \\
 0 &  0 &  0 \\
 1 &  2 &  1
\end{bmatrix}
$$

$$
M(x,y) = \sqrt{G_x^2 + G_y^2} \approx |G_x| + |G_y|
$$

A direção vetorial local da borda pode ser inferida por $\alpha(x,y) = \tan^{-1}(G_y / G_x)$.

In [ ]:
#| label: fig-03-sobel
#| fig-cap: "Extração de componentes espaciais direcionais e magnitude total do Gradiente de Sobel."
#| echo: true
#| output: true

sobelx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
magnitude = np.sqrt(sobelx**2 + sobely**2)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(np.abs(sobelx), cmap='gray')
ax[0].set_title("Sobel Horizontal (Gx)")
ax[0].axis('off')

ax[1].imshow(np.abs(sobely), cmap='gray')
ax[1].set_title("Sobel Vertical (Gy)")
ax[1].axis('off')

ax[2].imshow(magnitude, cmap='gray')
ax[2].set_title("Magnitude Combinada")
ax[2].axis('off')
plt.show()

### Detector de Bordas Ótimo de Canny

O algoritmo multiestágio de Canny baseia-se em três critérios fundamentais: baixa taxa de erro (capturar todas as bordas reais), localização precisa (proximidade mínima entre a borda real e detectada) e resposta única (uma única marcação por contorno). Seu fluxo computacional consiste em:

1. **Suavização Gaussiana:** Redução de variações espúrias locais de alta frequência;
2. **Cálculo de Gradiente:** Computação da magnitude e ângulo vetorial através do operador de Sobel;
3. **Supressão de Não-Máximos:** Varredura direcional de pixels para afinar contornos largos, retendo apenas picos locais locais de intensidade;
4. **Limiarização por Histerese:** Utilização de dois limiares distintos (alto e baixo). Pixels acima do limiar alto são validados como contornos fortes; pixels abaixo do limiar baixo são descartados. Pixels intermediários são confirmados se, e somente se, estiverem topologicamente conectados a um contorno forte.

In [ ]:
#| label: fig-03-canny
#| fig-cap: "Segmentação fina de contornos estruturais utilizando a técnica de Histerese de Canny."
#| echo: true
#| output: true

edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)

plt.figure(figsize=(7,7))
plt.imshow(edges, cmap='gray')
plt.title("Detector de Canny (Limiares: 100, 200)")
plt.axis('off')
plt.show()

## Análise Comparativa de Desempenho e Aplicações

A seleção adequada de um operador espacial exige o equilíbrio preciso entre a qualidade estatística do sinal resultante e a restrição de custo computacional imposta pelo hardware. A @tbl-03-filtros consolida o comportamento de cada abordagem abordada ao longo deste capítulo.

| Filtro | Categoria | Tipo de Ruído Alvo | Impacto em Bordas | Complexidade |
|---|---|---|---|---|
| **Média** | Linear (Passa-Baixa) | Gaussiano / Ruído Branco | Atenua e borra | $O(M \cdot N \cdot k^2)$ |
| **Gaussiano** | Linear (Passa-Baixa) | Ruído de Alta Frequência | Borrão preservado proporcional | $O(M \cdot N \cdot k^2)$ |
| **Mediana** | Não Linear (Ordem) | Impulsivo (Sal-e-Pimenta) | Preservação perfeita | $O(M \cdot N \cdot k^2 \log k)$ |
| **Laplaciano** | Linear (Passa-Alta) | Amplifica ruídos | Enfatiza transições | $O(M \cdot N \cdot k^2)$ |
| **Sobel** | Diferencial (Gradiente) | Suavização interna em eixos | Extrai magnitude vetorial | $O(M \cdot N \cdot k^2)$ |
| **Canny** | Multiestágio Ótimo | Eliminado por Gauss preliminar | Afinamento e conectividade | Alta |

: Síntese das características e comportamentos dos operadores espaciais do Capítulo 3. {#tbl-03-filtros}

## Resumo

Neste capítulo foram apresentados os fundamentos do processamento local de imagens no domínio espacial:

- **Manipulação de Pixels e Histogramas:** Processamento pontual através de álgebra matricial, mascaramento lógico bit-a-bit e realce por equalização global de distribuições de frequência.
- **Mecanismo de Convolução Espacial:** Formulação matemática discreta de máscaras móveis e estratégias de preenchimento para atenuação de artefatos de borda.
- **Suavização e Filtragem Passa-Baixa:** Aplicação dos operadores lineares de média e a distribuição isotrópica Gaussiana.
- **Atenuação de Ruído Não Linear:** O uso estratégico do filtro de ordem de mediana para isolar e restaurar imagens sob ruído impulsivo sal-e-pimenta.
- **Filtros de Nitidez e Detecção de Bordas:** Métodos de diferenciação baseados em gradientes de primeira ordem (Sobel), segunda ordem (Laplaciano) e os estágios avançados do detector ótimo de Canny.

O próximo capítulo estenderá esses conceitos para o **Domínio da Frequência**, abordando a Transformada de Fourier Bidimensional.

## 🤖 Uso do NotebookLM como Tutor Complementar

Para expandir o aprendizado deste capítulo com o projeto assistido por IA, sugere-se a exploração dos seguintes eixos temáticos na plataforma:
- Solicite simulações analíticas passo a passo sobre a convolução manual de pequenas matrizes digitais;
- Peça a distinção matemática formal e estrutural entre convolução e correlação cruzada no espaço;
- Peça uma análise detalhada sobre os impactos de variação dos limiares de histerese no detector de Canny em imagens médicas;
- Questione o comportamento e as limitações do filtro de mediana quando a probabilidade de ruído sal-e-pimenta excede 50% da cena.

## Lista de Exercícios

1. **(15%)** Explique a diferença teórica e prática entre os processos de Convolução e Correlação Cruzada. Em quais condições simétricas de um kernel os resultados numéricos finais de ambas as operações tornam-se rigorosamente idênticos?

2. **(15%)** Demonstre matematicamente por que a soma de todos os coeficientes pertencentes a uma máscara linear passa-alta (como o Laplaciano) deve totalizar zero, enquanto em um filtro passa-baixa a soma deve totalizar um.

3. **(20%)** Utilizando a imagem de teste carregada, implemente um script que adicione 5% de ruído sal-e-pimenta. Processe o resultado utilizando um filtro de média 3x3 e um filtro de mediana 3x3. Descreva qualitativamente o comportamento visual de ambos e justifique a disparidade de eficiência com base nas propriedades estatísticas dos operadores.

4. **(15%)** Descreva sucintamente o papel da etapa de "Supressão de Não-Máximos" no fluxo do algoritmo de detecção de contornos de Canny e explique o motivo pelo qual este estágio previne a ocorrência de falsas bordas duplicadas e largas.

5. **(20%)** Implemente, utilizando o contêiner do NumPy (sem suporte direto a funções prontas do OpenCV), uma função capaz de aplicar a técnica de *Unsharp Masking* com ganho ajustável $k$. Teste os resultados em uma imagem intencionalmente borrada e relate as alterações espectrais observadas na cena final.

6. **(15%)** Considere uma matriz de imagem contendo uma rampa de intensidade suave e linear de cinza e um ruído aleatório gaussiano de pequena magnitude sobreposto. Se aplicarmos o operador Sobel e o operador Laplaciano diretamente nesta cena, qual das respostas apresentará maior sensibilidade ao ruído de fundo? Justifique com base na ordem de derivação empregada por cada filtro.

## Parte Prática com Exercícios de Programação

### Objetivo deste Caderno

Os exercícios a seguir consolidam os conceitos estudados neste capítulo através de implementações práticas utilizando Python e as bibliotecas científicas de processamento NumPy e OpenCV.

### EP03_01 — Convolução Manual 2D

In [ ]:
# Escreva sua solução aqui


### EP03_02 — Filtro da Média

In [ ]:
# Escreva sua solução aqui


### EP03_03 — Filtro Gaussiano

In [ ]:
# Escreva sua solução aqui


### EP03_04 — Ruído Sal-e-Pimenta

In [ ]:
# Escreva sua solução aqui


### EP03_05 — Filtro da Mediana

In [ ]:
# Escreva sua solução aqui


### EP03_06 — Kernel de Nitidez

In [ ]:
# Escreva sua solução aqui


### EP03_07 — Detector de Bordas Sobel

In [ ]:
# Escreva sua solução aqui


### EP03_08 — Magnitude do Gradiente

In [ ]:
# Escreva sua solução aqui


### EP03_09 — Detector de Bordas Canny

In [ ]:
# Escreva sua solução aqui


### EP03_10 — Comparação entre Filtros

In [ ]:
# Escreva sua solução aqui
